# NSFnets 案例 3：Beltrami 流（3D 非定常）

**基于物理信息神经网络（PINN）求解不可压 Navier-Stokes 方程**

- 流动类型：Beltrami 流（a=d=1，Re=1，有精确解析解）
- 维度：3D 非定常 (x, y, z, t)，完整 NS 方程
- 网络结构：[4, 100×10, 4] — 10 个隐藏层，每层 100 个神经元
- 损失函数：α·L_initial + β·L_boundary + L_residual，α=β=100

Beltrami 流是 NS 方程少数已知的 3D 精确解之一，非常适合验证 PINN 在三维时空域上的精度。
无需外部数据——所有训练数据均由解析解生成。

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import time

from nsfnet_module import NSFnet3DUnsteady, relative_l2_error

np.random.seed(1234)
print('Imports complete.')

## 1. 解析解（Beltrami 流）

Beltrami 流是满足 NS 方程的精确三维非定常解（参数 a=d=1, Re=1）：

$$\begin{aligned}
u(x,y,z,t) &= -a\left[e^{ax}\sin(ay+dz) + e^{az}\cos(ax+dy)\right] e^{-d^2 t} \\
v(x,y,z,t) &= -a\left[e^{ay}\sin(az+dx) + e^{ax}\cos(ay+dz)\right] e^{-d^2 t} \\
w(x,y,z,t) &= -a\left[e^{az}\sin(ax+dy) + e^{ay}\cos(az+dx)\right] e^{-d^2 t} \\
p(x,y,z,t) &= -\frac{1}{2}a^2\left[e^{2ax}+e^{2ay}+e^{2az} + \text{交叉项}\right] e^{-2d^2 t}
\end{aligned}$$

In [ ]:
def beltrami_solution(x, y, z, t, a=1.0, d=1.0):
    """Analytic Beltrami flow solution.
    Returns: u, v, w, p (each same shape as inputs)
    """
    ex = np.exp(a * x); ey = np.exp(a * y); ez = np.exp(a * z)
    e2x = np.exp(2*a*x); e2y = np.exp(2*a*y); e2z = np.exp(2*a*z)
    edt = np.exp(-d**2 * t)
    e2dt = np.exp(-2 * d**2 * t)

    sx = np.sin(a*x + d*y); cx = np.cos(a*x + d*y)
    sy = np.sin(a*y + d*z); cy = np.cos(a*y + d*z)
    sz = np.sin(a*z + d*x); cz = np.cos(a*z + d*x)

    u = -a * (ex * sy + ez * cx) * edt
    v = -a * (ey * sz + ex * cy) * edt
    w = -a * (ez * sx + ey * cz) * edt

    p = -0.5 * a**2 * (
        e2x + e2y + e2z +
        2 * sx * cz * np.exp(a*(y+z)) +
        2 * sy * cx * np.exp(a*(z+x)) +
        2 * sz * cy * np.exp(a*(x+y))
    ) * e2dt

    return u, v, w, p

print('Beltrami solution function defined.')

## 2. 生成训练数据

计算域：x, y, z ∈ [-1, 1]³，t ∈ [0, 1]
- 边界配点：6 个面 × 900 点 × 11 时间步 = 59,400 点
- 初始条件：31³ = 29,791 个网格点（t=0）
- 内部配点：10,000 个随机时空点

In [ ]:
# Grid setup
x1 = np.linspace(-1, 1, 31, dtype=np.float32)
y1 = np.linspace(-1, 1, 31, dtype=np.float32)
z1 = np.linspace(-1, 1, 31, dtype=np.float32)
t1 = np.linspace(0, 1, 11, dtype=np.float32)

b0 = np.array([-1]*900, dtype=np.float32)
b1 = np.array([1]*900, dtype=np.float32)

# Build boundary face coordinates by tiling
xt = np.tile(x1[0:30], 30); yt = np.tile(y1[0:30], 30); zt = np.tile(z1[0:30], 30)
xt1 = np.tile(x1[1:31], 30); yt1 = np.tile(y1[1:31], 30); zt1 = np.tile(z1[1:31], 30)
xr = x1[0:30].repeat(30); yr = y1[0:30].repeat(30); zr = z1[0:30].repeat(30)
xr1 = x1[1:31].repeat(30); yr1 = y1[1:31].repeat(30); zr1 = z1[1:31].repeat(30)

# 6 faces × 900 points = 5400 boundary points per time step
train1x = np.concatenate([b1, b0, xt1, xt, xt1, xt], 0).repeat(t1.shape[0])
train1y = np.concatenate([yt, yt1, b1, b0, yr1, yr], 0).repeat(t1.shape[0])
train1z = np.concatenate([zr, zr1, zr, zr1, b1, b0], 0).repeat(t1.shape[0])
train1t = np.tile(t1, 5400)

# Boundary values from analytic solution
train1ub, train1vb, train1wb, train1pb = beltrami_solution(
    train1x, train1y, train1z, train1t)

# Reshape to (N, 1)
xb_train = train1x.reshape(-1,1).astype(np.float32)
yb_train = train1y.reshape(-1,1).astype(np.float32)
zb_train = train1z.reshape(-1,1).astype(np.float32)
tb_train = train1t.reshape(-1,1).astype(np.float32)
ub_train = train1ub.reshape(-1,1).astype(np.float32)
vb_train = train1vb.reshape(-1,1).astype(np.float32)
wb_train = train1wb.reshape(-1,1).astype(np.float32)

print(f'Boundary points: {xb_train.shape[0]} (5400×11 time steps)')

In [ ]:
# Initial condition (t=0)
x_0 = np.tile(x1, 31*31)
y_0 = np.tile(y1.repeat(31), 31)
z_0 = z1.repeat(31*31)
t_0 = np.zeros(x_0.shape[0], dtype=np.float32)

u_0, v_0, w_0, p_0 = beltrami_solution(x_0, y_0, z_0, t_0)

x0_train = x_0.reshape(-1,1).astype(np.float32)
y0_train = y_0.reshape(-1,1).astype(np.float32)
z0_train = z_0.reshape(-1,1).astype(np.float32)
t0_train = t_0.reshape(-1,1).astype(np.float32)
u0_train = u_0.reshape(-1,1).astype(np.float32)
v0_train = v_0.reshape(-1,1).astype(np.float32)
w0_train = w_0.reshape(-1,1).astype(np.float32)

print(f'Initial points: {x0_train.shape[0]} (31×31×31)')

In [ ]:
# Interior collocation points (random)
N_train = 10000
xx = (np.random.randint(31, size=N_train) / 15 - 1).astype(np.float32)
yy = (np.random.randint(31, size=N_train) / 15 - 1).astype(np.float32)
zz = (np.random.randint(31, size=N_train) / 15 - 1).astype(np.float32)
tt = (np.random.randint(11, size=N_train) / 10).astype(np.float32)

x_train = xx.reshape(-1,1)
y_train = yy.reshape(-1,1)
z_train = zz.reshape(-1,1)
t_train = tt.reshape(-1,1)

print(f'Interior points: {x_train.shape[0]}')

## 3. 构建模型

网络架构：输入 (x,y,z,t) → 100×10 隐藏层 → 输出 (u,v,w,p)
10 个隐藏层，每层 100 个神经元，约 9.2 万个可训练参数

In [ ]:
layers = [4] + [100]*10 + [4]
print(f'Layer structure: {layers}')

model3d = NSFnet3DUnsteady(
    x0_train, y0_train, z0_train, t0_train,
    u0_train, v0_train, w0_train,
    xb_train, yb_train, zb_train, tb_train,
    ub_train, vb_train, wb_train,
    x_train, y_train, z_train, t_train, layers,
    Re=1.0, alpha=100.0, beta=100.0
)

total_params = sum(int(np.prod(p.shape)) for p in model3d.net.trainable_params())
print(f'Trainable parameters: {total_params}')

## 4. 训练

**注意**：本案例约 9.2 万参数 + 10 层隐藏层，训练速度比 Case 1-2 慢。
建议先用少量迭代验证代码正确性，再进行完整训练。

In [ ]:
print('=== Phase 1: Adam LR=1e-3 ===')
model3d.adam_train(nIter=5000, learning_rate=1e-3, print_every=500)

In [ ]:
print('=== Phase 2: Adam LR=1e-4 ===')
model3d.adam_train(nIter=5000, learning_rate=1e-4, print_every=500)

In [ ]:
print('=== Phase 3: Adam LR=1e-5 ===')
model3d.adam_train(nIter=50000, learning_rate=1e-5, print_every=5000)

In [ ]:
print('=== Phase 4: Adam LR=1e-6 ===')
model3d.adam_train(nIter=50000, learning_rate=1e-6, print_every=5000)

In [ ]:
print('=== Phase 5: L-BFGS ===')
model3d.lbfgs_train(maxiter=50000)

## 5. 模型评估

在随机测试点上对比预测值与解析解

In [ ]:
# Test on random points
N_test = 1000
x_star = ((np.random.rand(N_test, 1) - 0.5) * 2).astype(np.float32)
y_star = ((np.random.rand(N_test, 1) - 0.5) * 2).astype(np.float32)
z_star = ((np.random.rand(N_test, 1) - 0.5) * 2).astype(np.float32)
t_star = (np.random.randint(11, size=(N_test, 1)) / 10).astype(np.float32)

# True values
u_star, v_star, w_star, p_star = beltrami_solution(x_star, y_star, z_star, t_star)

# Predict
t0 = time.time()
u_pred, v_pred, w_pred, p_pred = model3d.predict(x_star, y_star, z_star, t_star)
print(f'Prediction time: {time.time() - t0:.2f}s')

In [ ]:
# Relative L2 errors
error_u = relative_l2_error(u_pred, u_star)
error_v = relative_l2_error(v_pred, v_star)
error_w = relative_l2_error(w_pred, w_star)
error_p = relative_l2_error(p_pred, p_star)

print(f'Relative L2 Error — u: {error_u:.3e}')
print(f'Relative L2 Error — v: {error_v:.3e}')
print(f'Relative L2 Error — w: {error_w:.3e}')
print(f'Relative L2 Error — p: {error_p:.3e}')

print('\n=== Expected performance (original TF1 paper) ===')
print('Error u: ~1e-4 — 1e-3')
print('Error v: ~1e-4 — 1e-3')
print('Error w: ~1e-4 — 1e-3')
print('Error p: ~1e-3 — 1e-2')

## 总结

本 Notebook 实现了 3D 非定常 Beltrami 流案例：
- **解析解驱动**：所有训练数据由 Beltrami 精确解生成（a=d=1, Re=1）
- **3D+t 域**：x,y,z∈[-1,1]³，t∈[0,1]
- **深度网络**：10 层 × 100 神经元（约 9.2 万参数）
- **5.9 万边界点**：6 个面 × 900 点 × 11 时间步
- **2.9 万初始点**：31³ 完整网格（t=0 时刻）